In [1]:
from pathlib import Path
import pandas as pd, json
BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
RES = BASE / "results"
PAPER = BASE / "paper"
PAPER.mkdir(parents=True, exist_ok=True)

summary = pd.read_csv(RES / "flagging_rate_summary.csv") if (RES/"flagging_rate_summary.csv").exists() else pd.DataFrame()
decay = json.loads((RES/"decay_metrics.json").read_text()) if (RES/"decay_metrics.json").exists() else {}
hotspots = pd.read_csv(RES/"top20_hotspots.csv") if (RES/"top20_hotspots.csv").exists() else pd.DataFrame()
abstract = (BASE/'docs/project_abstract.md').read_text(encoding="utf-8")[:800] if (BASE/'docs/project_abstract.md').exists() else 'Abstract: We audit moderation sensitivity during the Ukraine crisis (MiDe22 EN01-EN10) vs baseline (Misc) using Perspective/HF proxies, neutral probes, and temporal decay analysis.'


In [3]:
paper = f"""# Crisis-Driven Moderation: Sensitivity Analysis (MiDe22 EN)

## Abstract
{abstract}

## 1. Introduction
Motivation per docs/project_proposal.md:15 — static models degrade during crises (No Time Like the Present [2], Temporal Bias [3]).

## 2. Method
- Data: MiDe22 EN n=5284, scoped Ukraine (EN01-10, n=1331) vs Misc (EN31-40, n=1391), cleaned 1000+1000, 50 neutral probes (25 generic + 25 injected, generic fixed to avoid 'embassy' contamination, injected via data-driven TF-IDF delta top-20).
- Auditing: Perspective API DEPRECATED — local HF cardiffnlp/twitter-roberta-base-offensive / unitary/toxic-bert with deterministic lexicon offline fallback (USE_HF=1 to enable HF), FR at 0.5/0.7/0.8.
- Drift: TF-IDF delta + log-odds + optional BERTopic, hotspot mapping to flagged tweets (data-driven, no hardcoded list).
- Decay: TF-IDF + LR (baseline-fit) and RF replication, test on crisis; mitigated mix (+20% crisis).

## 3. Results
Flagging Rate @0.7:
{summary[summary['threshold']==0.7].to_string(index=False) if not summary.empty else 'run Day03'}

Decay:
{json.dumps(decay, indent=2)}

Hotspots top 10:
{hotspots.head(10).to_string(index=False) if not hotspots.empty else 'run Day04'}

Figures: see ../figures/fig1_flagging_rate.png etc.

## 4. Discussion
Shifts imply over-flagging of legitimate crisis discourse; time-stratified mixing mitigates F1 decay. Limitations: English-only, topic-time confound, proxy thresholds.

## 5. References
See docs/project_proposal.md:47 + Toraman et al. 2024 MiDe22.

"""
(PAPER / "draft.md").write_text(paper, encoding="utf-8")
print(f"Draft -> {PAPER/'draft.md'}")


Draft -> C:\Users\phoen\Code\Repos\jupyter\sm-bias\paper\draft.md
